# Notebook 2: DistilBERT baseline
**Kaggle: Accelerator = GPU T4, Internet ON (downloads `distilbert-base-uncased`), attach Dataset `nli-data`.**

Protocol: train on SNLI 100K. Learning rate chosen on **SNLI validation only** (grid of 3, seed 42).
The best config is then run with 3 seeds (42, 43, 44; seed 42 is reused from the grid) and evaluated on
SNLI val/test, MNLI-matched, MNLI-mismatched. Test sets are only used after the config is frozen.
Fixed: 3 epochs, batch 32, warmup 10%, linear decay, weight decay 0.01, fp16, AdamW, grad-clip 1.0.
Expected runtime: ~5-8 min per run on a T4 (5 runs total), to be confirmed by the log.

In [1]:
import glob, json, os, sys, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup, set_seed

hits = glob.glob(os.environ.get("NLI_ROOT", "/kaggle/input") + "/**/nli_common.py", recursive=True)
assert hits, "nli_common.py not found: attach the 'nli-data' Dataset to this notebook"
ROOT = Path(hits[0]).parent
sys.path.insert(0, str(ROOT))
DATA = str(ROOT / "data")
from nli_common import LABELS, SEED, load_split, save_run

OUT = Path(os.environ.get("NLI_OUT", "/kaggle/working/results/distilbert")); OUT.mkdir(parents=True, exist_ok=True)
CKPT = Path("/kaggle/working/ckpt"); CKPT.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "Turn on a GPU accelerator (T4) in the notebook settings"
dev = "cuda"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

MODEL = "distilbert-base-uncased"
EPOCHS, BATCH, WARMUP, WD, CLIP = 3, 32, 0.1, 0.01, 1.0
LR_GRID, SEEDS = [2e-5, 3e-5, 5e-5], [42, 43, 44]
MAX_LEN_TRAIN = 128
MAX_LEN_EVAL = {"snli_val": 128, "snli_test": 128, "mnli_m": 256, "mnli_mm": 256}   # MNLI premises are longer

train = load_split("train_100k", DATA)
EVALS = {n: load_split(n, DATA) for n in ["snli_val", "snli_test", "mnli_m", "mnli_mm"]}
val = EVALS["snli_val"]
print({"train": len(train), **{k: len(v) for k, v in EVALS.items()}})

GPU: Tesla T4 | torch 2.11.0+cu128
{'train': 100000, 'snli_val': 9842, 'snli_test': 9824, 'mnli_m': 9815, 'mnli_mm': 9832}


## Data and helper functions

In [2]:
tok = AutoTokenizer.from_pretrained(MODEL)

def encode(df, max_len):
    enc = tok(df["premise"].tolist(), df["hypothesis"].tolist(), truncation=True, max_length=max_len)
    return enc["input_ids"], enc["attention_mask"]

ids, am = encode(train, MAX_LEN_TRAIN)
train_items = [({"input_ids": i, "attention_mask": a}, int(l)) for i, a, l in zip(ids, am, train["label"])]
print("train token length: mean %.1f, max %d, truncated: %d" % (np.mean([len(i) for i in ids]), max(len(i) for i in ids),
                                                              sum(len(i) >= MAX_LEN_TRAIN for i in ids)))

def collate(batch):
    feats = dict(tok.pad([b[0] for b in batch], return_tensors="pt"))
    feats["labels"] = torch.tensor([b[1] for b in batch])
    return feats

@torch.no_grad()
def predict(model, df, max_len, bs=128):
    """Softmax probabilities in the original row order. Sorted by length internally for speed."""
    model.eval()
    ids, am = encode(df, max_len)
    order = np.argsort([len(x) for x in ids])
    out = np.zeros((len(df), 3), dtype=np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        feats = tok.pad([{"input_ids": ids[j], "attention_mask": am[j]} for j in idx], return_tensors="pt")
        feats = {k: v.to(dev) for k, v in feats.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**feats).logits
        out[idx] = torch.softmax(logits.float(), -1).cpu().numpy()
    return out

def new_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=3, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)}).to(dev)

def train_run(lr, seed):
    set_seed(seed)
    model = new_model()
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(train_items, batch_size=BATCH, shuffle=True, generator=g, collate_fn=collate, num_workers=2)
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")   # DistilBERT names: sa_layer_norm / output_layer_norm
    groups = [{"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)], "weight_decay": WD},
              {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=lr)
    total = EPOCHS * len(loader)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
    scaler = torch.cuda.amp.GradScaler()
    torch.cuda.reset_peak_memory_stats()
    hist, train_time = [], 0.0
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); loss_sum = 0.0
        for batch in loader:
            batch = {k: v.to(dev, non_blocking=True) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            loss_sum += loss.item()
        ep_time = time.time() - t0; train_time += ep_time
        acc = float((predict(model, val, 128).argmax(1) == val["label"].to_numpy()).mean())
        hist.append({"epoch": ep + 1, "train_loss": loss_sum / len(loader), "val_acc": acc, "epoch_s": round(ep_time)})
        print(f"  lr={lr:g} seed={seed} epoch {ep + 1}/{EPOCHS}  loss={loss_sum / len(loader):.4f}  val acc={acc:.4f}  ({ep_time:.0f}s)")
    return model, {"history": hist, "train_time_s": train_time, "peak_gpu_mem_gb": torch.cuda.max_memory_allocated() / 1e9}

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

train token length: mean 26.1, max 100, truncated: 0


## 1. Learning-rate grid (seed 42, validation only)

In [3]:
grid, best = [], {"val_acc": -1}
for lr in LR_GRID:
    model, info = train_run(lr, 42)
    acc = info["history"][-1]["val_acc"]
    grid.append({"lr": lr, "val_acc_final_epoch": acc, "train_time_s": round(info["train_time_s"])})
    if acc > best["val_acc"]:
        best = {"lr": lr, "val_acc": acc, "info": info}
        torch.save(model.state_dict(), CKPT / "best_grid.pt")
    del model; torch.cuda.empty_cache()
grid_df = pd.DataFrame(grid); grid_df.to_csv(OUT / "distilbert_lr_grid.csv", index=False)
BEST_LR = best["lr"]
print(grid_df, "\nSelected lr (by validation accuracy):", BEST_LR)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_23/2184644792.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


  lr=2e-05 seed=42 epoch 1/3  loss=0.6273  val acc=0.8270  (185s)
  lr=2e-05 seed=42 epoch 2/3  loss=0.4075  val acc=0.8459  (184s)
  lr=2e-05 seed=42 epoch 3/3  loss=0.3136  val acc=0.8498  (183s)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  lr=3e-05 seed=42 epoch 1/3  loss=0.6085  val acc=0.8354  (183s)
  lr=3e-05 seed=42 epoch 2/3  loss=0.3778  val acc=0.8515  (184s)
  lr=3e-05 seed=42 epoch 3/3  loss=0.2639  val acc=0.8568  (183s)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  lr=5e-05 seed=42 epoch 1/3  loss=0.5951  val acc=0.8412  (183s)
  lr=5e-05 seed=42 epoch 2/3  loss=0.3578  val acc=0.8590  (183s)
  lr=5e-05 seed=42 epoch 3/3  loss=0.2209  val acc=0.8588  (183s)
        lr  val_acc_final_epoch  train_time_s
0  0.00002             0.849827           552
1  0.00003             0.856838           550
2  0.00005             0.858769           550 
Selected lr (by validation accuracy): 5e-05


## 2. Final runs: best lr, seeds 42 (reused from grid), 43, 44; evaluate on all sets

In [4]:
def evaluate_and_save(model, seed, info):
    name = f"distilbert_s{seed}"
    config = {"model": MODEL, "train_file": "snli_train_100k", "n_train": len(train), "seed": seed, "lr": BEST_LR,
              "epochs": EPOCHS, "batch_size": BATCH, "warmup_frac": WARMUP, "weight_decay": WD, "scheduler": "linear",
              "grad_clip": CLIP, "precision": "fp16 autocast", "max_len_train": MAX_LEN_TRAIN, "max_len_eval": MAX_LEN_EVAL,
              "lr_grid": LR_GRID, "history": info["history"]}
    n_params = sum(p.numel() for p in model.parameters())
    for es, df in EVALS.items():
        t0 = time.time(); torch.cuda.synchronize()
        probs = predict(model, df, MAX_LEN_EVAL[es]); torch.cuda.synchronize()
        dt = time.time() - t0
        save_run(OUT, name, es, df, probs, config,
                 {"train_time_s": info["train_time_s"], "peak_gpu_mem_gb": info["peak_gpu_mem_gb"], "n_params": n_params,
                  "inference_s": dt, "inference_examples_per_s": len(df) / dt})

for seed in SEEDS:
    if seed == 42:
        model = new_model(); model.load_state_dict(torch.load(CKPT / "best_grid.pt")); info = best["info"]
        model.save_pretrained(OUT / "distilbert_s42_model"); tok.save_pretrained(OUT / "distilbert_s42_model")   # for the demo
    else:
        model, info = train_run(BEST_LR, seed)
    evaluate_and_save(model, seed, info)
    del model; torch.cuda.empty_cache()
os.remove(CKPT / "best_grid.pt")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[distilbert_s42 | snli_val] acc=0.8588 macroF1=0.8584 per-class F1=[0.881, 0.821, 0.873]
[distilbert_s42 | snli_test] acc=0.8585 macroF1=0.8580 per-class F1=[0.881, 0.815, 0.878]
[distilbert_s42 | mnli_m] acc=0.6278 macroF1=0.6245 per-class F1=[0.674, 0.582, 0.617]
[distilbert_s42 | mnli_mm] acc=0.6477 macroF1=0.6437 per-class F1=[0.694, 0.587, 0.651]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_23/2184644792.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


  lr=5e-05 seed=43 epoch 1/3  loss=0.5974  val acc=0.8425  (183s)
  lr=5e-05 seed=43 epoch 2/3  loss=0.3574  val acc=0.8589  (183s)
  lr=5e-05 seed=43 epoch 3/3  loss=0.2219  val acc=0.8609  (183s)
[distilbert_s43 | snli_val] acc=0.8609 macroF1=0.8607 per-class F1=[0.883, 0.825, 0.874]
[distilbert_s43 | snli_test] acc=0.8576 macroF1=0.8573 per-class F1=[0.881, 0.815, 0.876]
[distilbert_s43 | mnli_m] acc=0.6310 macroF1=0.6280 per-class F1=[0.682, 0.599, 0.603]
[distilbert_s43 | mnli_mm] acc=0.6555 macroF1=0.6526 per-class F1=[0.705, 0.604, 0.649]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_23/2184644792.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


  lr=5e-05 seed=44 epoch 1/3  loss=0.5965  val acc=0.8402  (184s)
  lr=5e-05 seed=44 epoch 2/3  loss=0.3624  val acc=0.8566  (184s)
  lr=5e-05 seed=44 epoch 3/3  loss=0.2264  val acc=0.8587  (184s)
[distilbert_s44 | snli_val] acc=0.8587 macroF1=0.8585 per-class F1=[0.88, 0.818, 0.877]
[distilbert_s44 | snli_test] acc=0.8584 macroF1=0.8580 per-class F1=[0.882, 0.816, 0.876]
[distilbert_s44 | mnli_m] acc=0.6250 macroF1=0.6232 per-class F1=[0.668, 0.605, 0.596]
[distilbert_s44 | mnli_mm] acc=0.6484 macroF1=0.6472 per-class F1=[0.692, 0.609, 0.641]


## 3. Summary: mean ± std over seeds

In [5]:
rows = []
for es in EVALS:
    accs, f1s = [], []
    for seed in SEEDS:
        m = json.load(open(OUT / f"results_distilbert_s{seed}_{es}.json"))["metrics"]
        accs.append(m["accuracy"]); f1s.append(m["f1_macro"])
    rows.append({"eval_set": es, "acc_mean": np.mean(accs), "acc_std": np.std(accs, ddof=1),
                 "macroF1_mean": np.mean(f1s), "macroF1_std": np.std(f1s, ddof=1), "per_seed_acc": [round(a, 4) for a in accs]})
summ = pd.DataFrame(rows); summ.to_csv(OUT / "distilbert_summary.csv", index=False)
print(summ.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nFiles:", sorted(os.listdir(OUT)))

 eval_set  acc_mean  acc_std  macroF1_mean  macroF1_std             per_seed_acc
 snli_val    0.8594   0.0013        0.8592       0.0013 [0.8588, 0.8609, 0.8587]
snli_test    0.8582   0.0005        0.8578       0.0004 [0.8585, 0.8576, 0.8584]
   mnli_m    0.6279   0.0030        0.6252       0.0025   [0.6278, 0.631, 0.625]
  mnli_mm    0.6505   0.0043        0.6478       0.0045 [0.6477, 0.6555, 0.6484]

Files: ['distilbert_lr_grid.csv', 'distilbert_s42_model', 'distilbert_summary.csv', 'preds_distilbert_s42_mnli_m.csv', 'preds_distilbert_s42_mnli_mm.csv', 'preds_distilbert_s42_snli_test.csv', 'preds_distilbert_s42_snli_val.csv', 'preds_distilbert_s43_mnli_m.csv', 'preds_distilbert_s43_mnli_mm.csv', 'preds_distilbert_s43_snli_test.csv', 'preds_distilbert_s43_snli_val.csv', 'preds_distilbert_s44_mnli_m.csv', 'preds_distilbert_s44_mnli_mm.csv', 'preds_distilbert_s44_snli_test.csv', 'preds_distilbert_s44_snli_val.csv', 'results_distilbert_s42_mnli_m.json', 'results_distilbert_s42_mnli_mm.js